In [ ]:
import polars as pl
import numpy as np
from pathlib import Path
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import statsmodels.api as stm
import scipy.stats as st
from matplotlib.colors import Normalize
from datetime import datetime as dt


from piepy.psychophysics.wheel.detection.wheelDetectionExperimentHub import WheelDetectionExperimentHub
from piepy.core.data_functions import make_subsets
from piepy.plotters.plotting_utils import set_style
from piepy.psychophysics.wheel.detection.wheelDetectionGroupedAggregator import WheelDetectionGroupedAggregator

In [ ]:
DATA_PATH = Path.cwd().parents[0] / "260410_Ncomm_inhibition_data.parquet"

In [ ]:
hub = WheelDetectionExperimentHub()
# load from session list
# hub.set_data(all_sessions,
#              load_sessions=True,
#              make_summary=True)
# hub.data.write_parquet("250206_experiment_data.parquet")

# load from saved data
all_data = pl.read_parquet(DATA_PATH)
hub.set_data(all_data,
             load_sessions=True,
             make_summary=True)

In [ ]:
areas = ["V1","HVA","LM","AL","RL","PM","AM"]
stim_combination = "0.04cpd_8.0Hz"
df = hub.filter_by_areas(areas,
                    strict_performance=True,
                    stim_combination=stim_combination,
                    isCNO=False)
# df = df.filter(~pl.col("animalid").is_in(["KC141"]))
hub.make_summary_data(df)

In [ ]:
@np.vectorize
def lenient_div(num: np.ndarray, den: np.ndarray) -> np.ndarray:
    """returns the numerator if denominator is zero, otherwise does regular division

    Args:
        num (Number | np.ndarray): Numerator
        den (Number | np.ndarray): Denominator

    Returns:
        Number | np.ndarray: Division result
    """
    return float(den and num / den or num)

In [ ]:
AREA_COLORS={"V1":"#000000",
             "HVA":"#C7C7C7",
             "LM":"#EC1D23",
             "AL":"#009344",
             "RL":"#26A9E0",
             "PM":"#FAAF41",
             "AM":"#EC297A"}

cmm = 1/2.54
set_style("print")
f,ax = plt.subplots(1,2,
                    figsize=(20*cmm,12*cmm))

do_bsi = True
do_norm_delta_rt = True

# colormap stuff
cmap = "viridis"
normalizer = Normalize(5, 10)
im = cm.ScalarMappable(norm=normalizer, cmap=cmap)

aggregator = WheelDetectionGroupedAggregator()
aggregator.set_data(data=df)
aggregator.group_data(group_by = ["contrast", "stim_type", "stim_side", "area", "animalid", "opto_pattern"])
aggregator.calculate_hit_rates()
aggregator.calculate_opto_pvalues()

plot_data = aggregator.grouped_data.drop_nulls("contrast").filter(pl.col("stim_side")!="ipsi")
plot_data = plot_data.filter(pl.col("contrast").is_in([0.125, 0.5]))

# manually add baselines by joining
# catch_trials = plot_data.filter(
#     (pl.col("contrast") == 0) & (pl.col("opto_pattern") == -1)
# ).select(["animalid", "area", "contrast", "opto_pattern", "hit_count", "count"])
# # get baseline_hr
# catch_trials = catch_trials.with_columns(
#     (pl.col("hit_count") / pl.col("count")).alias("baseline_hr")
# )

# plot_data = plot_data.join(
#         catch_trials.select(["animalid", "area", "baseline_hr"]),
#         how="inner",
#         left_on=["animalid", "area"],
#         right_on=["animalid", "area"],
#     )

val_mat = np.zeros((plot_data.n_unique('area'),
                    5, # [mean_hr,sem_hr,mean_rt,sem_rt,n_animal]
                    plot_data.n_unique(['contrast'])))
val_mat[:] = np.nan

for filt_tup in make_subsets(plot_data,["contrast"],start_enumerate=0):
    filt_df = filt_tup[-1]
    aid_arr = []
    for area_tup in make_subsets(filt_df, ["area"], start_enumerate=0):
        area_df = area_tup[-1]
        aid_arr.append(area_tup[1])
        hr_arr = []
        rt_arr = []
        for stim_tup in make_subsets(area_df,["animalid"],start_enumerate=0):
            stim_df = stim_tup[-1].sort(["opto_pattern"])
            
            if not do_bsi:
                hr = 100*(stim_df[1,"hit_rate"] - stim_df[0,"hit_rate"])
            else:
                _top = stim_df[0,"hit_rate"] - stim_df[1,"hit_rate"]
                hr = 100*lenient_div(_top, stim_df[0,"hit_rate"])
            
            opto_hit_count = stim_df[1,"hit_count"]
            if not do_norm_delta_rt:
                if stim_df[1,"median_hit_reaction_times"] is not None:
                    delta_rt = stim_df[1,"median_hit_reaction_times"] - stim_df[0,"median_hit_reaction_times"]
                else:
                    delta_rt = np.nan
            else:
                if stim_df[1,"median_hit_reaction_times"] is not None:
                    _top = stim_df[1,"median_hit_reaction_times"] - stim_df[0,"median_hit_reaction_times"]
                    # _bottom = 1000 - stim_df[0,"median_hit_reaction_times"]
                    # delta_rt = 100*lenient_div(_top, _bottom)
                    delta_rt = stim_df[1,"median_hit_reaction_times"] - stim_df[0,"median_hit_reaction_times"]
                else:
                    delta_rt = np.nan
            
            hr_arr.append(hr)
            rt_arr.append(delta_rt)

        # take average
        mean_hr = np.nanmean(hr_arr)
        mean_rt = np.nanmean(rt_arr)
        sem_hr = st.sem(hr_arr,nan_policy="omit")
        sem_rt = st.sem(rt_arr,nan_policy="omit")
        
        val_mat[area_tup[0],:,filt_tup[0]] = [mean_hr,sem_hr,mean_rt,sem_rt,area_df.n_unique("animalid")]
    print(aid_arr)


c = ["#C7C7C7", "#313131"]    
for d in range(val_mat.shape[2]):
    contrast_slice = val_mat[:,:,d]
    
    # sort 
    aid_arr_sorted = [aid_arr[ii] for ii in contrast_slice[:, 2].argsort()]
    contrast_slice = contrast_slice[contrast_slice[:, 2].argsort()]
    
    _rt = contrast_slice[:,2]
    _hr = contrast_slice[:,0]
    _sem_hr = contrast_slice[:,1]
    _sem_rt = contrast_slice[:,3]
    
    clr = [c[d] for _ in contrast_slice]
    _area_colors = [AREA_COLORS[ii] for ii in aid_arr_sorted]
    ax[d].scatter(_rt,
                  _hr,
                  c=_area_colors,
                #   edgecolors = clr[d],
                  linewidths=1)
    
    # sem hr
    ax[d].vlines(_rt,
                 ymin=_hr-_sem_hr,
                 ymax=_hr+_sem_hr,
                 colors=_area_colors)
    
    # sem rt
    ax[d].hlines(_hr,
                 xmin=_rt-_sem_rt,
                 xmax=_rt+_sem_rt,
                 colors=_area_colors)
    
    for ai, coord in enumerate(zip(_rt,_hr)):
        ax[d].text(coord[0]+1,coord[1]+1, aid_arr_sorted[ai])

    non_nan = np.invert(np.isnan(_rt))
    
    X = stm.add_constant(contrast_slice[non_nan,2])
    ols_model = stm.OLS(contrast_slice[non_nan,0], X)
    est = ols_model.fit()
    out = est.conf_int(alpha=0.05, cols=None)

    y_pred = est.predict(X)
    x_pred = contrast_slice[non_nan,2]
    slope, intercept, r_value, p_value, std_err = st.linregress(contrast_slice[non_nan,2], contrast_slice[non_nan,0])


    ax[d].plot(x_pred,y_pred,color=clr[0],label=f"R2={r_value:.2}")

    pred = est.get_prediction(X).summary_frame()
    
    ax[d].fill_between(x=x_pred,
                    y1=pred['mean_ci_lower'].tolist(),
                    y2=pred['mean_ci_upper'].tolist(),
                    linewidth=0,
                    color=clr[0],
                    alpha=0.3)
    
    # ax.plot(x_pred,pred['mean_ci_lower'],linestyle='--',color=clr[0])
    # ax.plot(x_pred,pred['mean_ci_upper'],linestyle='--',color=clr[0])
    
    ax[d].legend(frameon=False)
    ax[d].set_xlabel(r"$\Delta$Reaction time (ms)")
    ax[d].axhline(50,c="k",linestyle=":")
    
    ax[d].set_yticks([0,25,50,75,100])
    ax[d].axvline(50,c="k",linestyle=":")
    ax[d].set_xticks([0,50,100,150,200,250,300,350])
    if not do_bsi:
        ax[d].set_ylabel(r"$\Delta$Hit rate (%)")
    else:
        ax[d].set_ylabel("SI (%)")
